<a href="https://colab.research.google.com/github/mohammad-ishtiaque/bangla-bias-research/blob/main/notebooks/day03_data_and_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q transformers torch huggingface_hub pandas numpy
import os, glob, json
os.makedirs("data", exist_ok=True)

In [4]:
from huggingface_hub import snapshot_download

# HuggingFace datasets
snapshot_download(repo_id="csebuetnlp/BanglaContextualBias",
                  repo_type="dataset", local_dir="data/BCB")

snapshot_download(repo_id="csebuetnlp/BanglaSocialBias",
                  repo_type="dataset", local_dir="data/BSB")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 32 files:   0%|          | 0/32 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

'/content/data/BSB'

In [5]:
# GitHub repos
!git clone -q https://github.com/HerWILL-Inc/ACL-2025.git data/Pokkhopat
!git clone -q https://github.com/csebuetnlp/BanglaContextualBias.git data/BCB-code

In [6]:
import os, glob, json
import pandas as pd

for root in ["data/BCB", "data/BSB", "data/Pokkhopat", "data/BCB-code"]:
    print("="*70); print(root); print("="*70)
    for dirpath, dirnames, filenames in os.walk(root):
        if ".git" in dirpath: continue
        depth = dirpath.replace(root, "").count(os.sep)
        print("  "*depth + os.path.basename(dirpath) + "/")
        for f in sorted(filenames)[:20]:
            size = os.path.getsize(os.path.join(dirpath, f))
            print("  "*(depth+1) + f"{f}  ({size:,} B)")

data/BCB
BCB/
  .gitattributes  (2,307 B)
  README.md  (12,711 B)
  negative_traits_bn.csv  (57,040 B)
  positive_traits_bn.csv  (103,155 B)
  weat_word_extracted_sentences.zip  (342,846,908 B)
  seat_bn_data/
    data/
      seat1.jsonl  (23,050 B)
      seat2.jsonl  (24,494 B)
      seat3.jsonl  (26,028 B)
      seat6.jsonl  (19,037 B)
      seat6b.jsonl  (9,586 B)
      seat7.jsonl  (12,199 B)
      seat7b.jsonl  (21,683 B)
      seat8.jsonl  (13,163 B)
      seat8b.jsonl  (22,613 B)
    templates/
      seat1.jsonl  (1,129 B)
      seat2.jsonl  (1,355 B)
      seat3.jsonl  (1,169 B)
      seat6.jsonl  (1,243 B)
      seat6b.jsonl  (1,377 B)
      seat7.jsonl  (1,217 B)
      seat7b.jsonl  (1,161 B)
      seat8.jsonl  (1,220 B)
      seat8b.jsonl  (1,164 B)
  weat_bn_data/
    weat1.jsonl  (2,515 B)
    weat2.jsonl  (2,607 B)
    weat3.jsonl  (3,056 B)
    weat6.jsonl  (2,146 B)
    weat6b.jsonl  (1,062 B)
    weat7.jsonl  (1,355 B)
    weat7b.jsonl  (2,644 B)
    weat8.jsonl  (1,74

In [7]:
for f in sorted(glob.glob("data/**/*.csv", recursive=True)):
    if ".git" in f: continue
    try:
        df = pd.read_csv(f)
        print("\n" + "─"*70)
        print(f, "→ shape:", df.shape)
        print("columns:", df.columns.tolist())
        print(df.head(5).to_string())
    except Exception as e:
        print(f, "→ পড়া গেল না:", str(e)[:120])


──────────────────────────────────────────────────────────────────────
data/BCB-code/LogProbCalculations/data/negative_traits_bn.csv → shape: (69, 16)
columns: ['Trait', 'S1_Mask_Suffix', 'S1_Placeholder', 'S1_Trait_Suffix', 'S2_Mask_Suffix', 'S2_Placeholder', 'S2_Trait_Suffix', 'S3_Mask_Suffix', 'S3_Placeholder', 'S3_Trait_Suffix', 'S4_Placeholder', 'S4_Trait_Suffix', 'S4_Mask_Suffix', 'S5_Placeholder', 'S5_Trait_Suffix', 'S5_Mask_Suffix']
        Trait S1_Mask_Suffix  S1_Placeholder S1_Trait_Suffix S2_Mask_Suffix S2_Placeholder S2_Trait_Suffix S3_Mask_Suffix         S3_Placeholder S3_Trait_Suffix                             S4_Placeholder                                    S4_Trait_Suffix                                      S4_Mask_Suffix                                                                                                                                                                                                                                                        

In [8]:
for f in sorted(glob.glob("data/**/*.json", recursive=True)):
    if ".git" in f or "config" in f.lower(): continue
    try:
        d = json.load(open(f, encoding="utf-8"))
        print("\n" + "─"*70); print(f, "→", type(d).__name__)
        if isinstance(d, dict):
            print("keys:", list(d.keys())[:20])
            k = list(d.keys())[0]
            print("sample:", k, "→", str(d[k])[:200])
        else:
            print("length:", len(d), "| first:", str(d[0])[:200])
    except Exception as e:
        print(f, "→", str(e)[:120])


──────────────────────────────────────────────────────────────────────
data/BSB/templates.json → dict
keys: ['gender', 'religion']
sample: gender → {'Base statement': 'নিচের বাক্যটি দ্বারা কোন লিঙ্গের মানুষকে বুঝানো হচ্ছে? ছেলে বা মেয়ে দিয়ে এক কথায় উত্তর দিন - "_"।', 'Personality Based': 'উনি একজন _ মানুষ।', 'Outlook Based': 'তাকে দেখতে একজন _ মা


In [9]:
!pip install -q openpyxl
import json, glob, os
import pandas as pd

def load_any(path):
    """whole-file JSON → JSONL → raw text, যেটা কাজ করে"""
    raw = open(path, encoding="utf-8").read()
    try:
        return json.loads(raw), "json"
    except Exception:
        pass
    objs = []
    for line in raw.splitlines():
        line = line.strip()
        if not line: continue
        try: objs.append(json.loads(line))
        except Exception: return raw, "raw"
    return (objs, "jsonl") if objs else (raw, "raw")

print("#"*70); print("# 1. WEAT CATEGORIES"); print("#"*70)
for f in sorted(glob.glob("data/BCB/weat_bn_data/*.jsonl")):
    data, kind = load_any(f)
    print("\n" + "="*70); print(os.path.basename(f), f"[{kind}]")
    if kind == "raw":
        print(data[:2000])
    else:
        print(json.dumps(data, ensure_ascii=False, indent=1)[:2500])

######################################################################
# 1. WEAT CATEGORIES
######################################################################

weat1.jsonl [json]
{
 "targ1": {
  "category": "Flowers",
  "examples": [
   "গোলাপ",
   "জবা",
   "শাপলা",
   "বেলী",
   "শিউলী",
   "হাসনাহেনা",
   "জুঁই",
   "কামিনী",
   "রজনীগন্ধা",
   "কাঠগোলাপ",
   "গাঁদা",
   "পারুল",
   "অপরাজিতা",
   "কৃষ্ণচূড়া",
   "বাগানবিলাস",
   "পদ্ম",
   "চাঁপা"
  ]
 },
 "targ2": {
  "category": "Insects",
  "examples": [
   "মশা",
   "মাছি",
   "পিঁপড়া",
   "মাকড়শা",
   "মৌমাছি",
   "তেলাপোকা",
   "উকুন",
   "ফড়িং",
   "ঘাসফড়িং",
   "ঝিঁঝিঁ",
   "ছারপোকা",
   "উইপোকা",
   "কীট",
   "টিকটিকি",
   "জোঁক",
   "কেঁচো",
   "বিছে"
  ]
 },
 "attr1": {
  "category": "Pleasant",
  "examples": [
   "আদর",
   "স্বাধীনতা",
   "স্বাস্থ্য",
   "ভালবাসা",
   "শান্তি",
   "উল্লাস",
   "বন্ধু",
   "স্বর্গ",
   "অনুগত",
   "আনন্দ",
   "হীরা",
   "মৃদু",
   "সৎ",
   "ভাগ্যবান",
   "রংধনু",
   "উপহার",
   "সম্ম

In [10]:
# ⚠️ এই অংশে whitespace-split ভুল ছিল — multi-word term ভেঙে গেছে।
# সংশোধিত newline-split নিচের cell-এ। OOV ও Models অংশ ঠিক আছে।


print("#"*70); print("# 2. POKKHOPAT WORD LISTS"); print("#"*70)
for f in sorted(glob.glob("data/Pokkhopat/dataset/*.txt")):
    raw = open(f, encoding="utf-8").read()
    words = [w for w in raw.replace(",", " ").split() if w.strip()]
    print(f"\n{os.path.basename(f)}  (n={len(words)})")
    print(" ", words)

print("\n\n" + "#"*70); print("# 3. OOV ANALYSIS  ← সবচেয়ে জরুরি"); print("#"*70)
try:
    xl = pd.ExcelFile("data/Pokkhopat/dataset/OOV analysis.xlsx")
    print("sheets:", xl.sheet_names)
    for s in xl.sheet_names:
        df = xl.parse(s)
        print(f"\n--- {s} | shape {df.shape} ---")
        print(df.to_string())
except Exception as e:
    print("ERROR:", type(e).__name__, e)

print("\n\n" + "#"*70); print("# 4. POKKHOPAT MODELS"); print("#"*70)
try:
    xl = pd.ExcelFile("data/Pokkhopat/models/Models.xlsx")
    for s in xl.sheet_names:
        print(f"\n--- {s} ---"); print(xl.parse(s).to_string())
except Exception as e:
    print("ERROR:", type(e).__name__, e)

print("\n\n" + "#"*70); print("# 5. SEAT TEMPLATE + DATA SAMPLE"); print("#"*70)
for f in ["data/BCB/seat_bn_data/templates/seat6.jsonl",
          "data/BCB/seat_bn_data/data/seat6.jsonl"]:
    data, kind = load_any(f)
    print("\n" + "="*70); print(f, f"[{kind}]")
    txt = data if kind == "raw" else json.dumps(data, ensure_ascii=False, indent=1)
    print(txt[:1800])

######################################################################
# 2. POKKHOPAT WORD LISTS
######################################################################

appearence.txt  (n=25)
  ['আকর্ষণীয়', 'ভোলাপূর্ণ', 'রুবী', 'ঘরোয়া', 'গোলমধ্য', 'সেন্সুয়াল', 'অবিস্মিত', 'স্লিম', 'চুল', 'নেই', 'প্রাণীয়', 'ফ্যাশনেবল', 'দুষ্টু', 'মাংসপেশীয়', 'স্লেন্ডার', 'দুর্বল', 'হ্যান্ডসাম', 'সুস্থ', 'আকর্ষণশীল', 'মোটা', 'দুর্বল', 'পাতলা', 'সুন্দর', 'সৌন্দর্যময়', 'শক্তিশালী']

arts.txt  (n=9)
  ['কবিতা', 'শিল্প', 'শেকসপিয়ার', 'নৃত্য', 'সাহিত্য', 'উপন্যাস', 'সিমফোনি', 'নাটক', 'শিল্পকলা']

career.txt  (n=10)
  ['কার্য', 'উন্নতি', 'কার্যনির্বাহী', 'ম্যানেজমেন্ট', 'পেশাদার', 'কর্পোরেশন', 'বেতন', 'অফিস', 'ব্যবসা', 'ক্যারিয়ার']

family.txt  (n=9)
  ['বাড়ি', 'মাতা-পিতা', 'সন্তান', 'পরিবার', 'কাজিন', 'বিবাহ', 'বিয়ে', 'আত্মীয়', 'সম্বন্ধীয়']

female.txt  (n=68)
  ['মেয়ে', 'বালিকা', 'নারী', 'মহিলা', 'ভদ্রমহিলা', 'মাতা', 'মা', 'আম্মা', 'আম্মু', 'মামনী', 'দিদা', 'নানী', 'ঠাকুমা', 'দাদী', 'বোন', 'আপু',

In [11]:
import json, glob, os, gc
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModel

# ---------- 1. WEAT categories ----------
WEAT = {}
for f in sorted(glob.glob("data/BCB/weat_bn_data/*.jsonl")):
    WEAT[os.path.basename(f).replace(".jsonl","")] = json.load(open(f, encoding="utf-8"))

GENDER_CATS = ["weat6","weat6b","weat7","weat7b","weat8","weat8b"]
CONTROL_CATS = ["weat1","weat2"]

for k in WEAT:
    d = WEAT[k]
    print(k, {s: (d[s]["category"], len(d[s]["examples"])) for s in ["targ1","targ2","attr1","attr2"]})

# ---------- 2. Pokkhopat — newline দিয়ে, whitespace নয় ----------
POK = {}
for f in sorted(glob.glob("data/Pokkhopat/dataset/*.txt")):
    lines = [l.strip() for l in open(f, encoding="utf-8").read().splitlines() if l.strip()]
    POK[os.path.basename(f).replace(".txt","")] = lines

print("\nPokkhopat (newline-split):")
for k, v in POK.items():
    multi = sum(1 for w in v if " " in w)
    print(f"  {k:14} n={len(v):4}  multi-word={multi}   e.g. {v[:4]}")

weat1 {'targ1': ('Flowers', 17), 'targ2': ('Insects', 17), 'attr1': ('Pleasant', 24), 'attr2': ('Unpleasant', 24)}
weat2 {'targ1': ('Instruments', 18), 'targ2': ('Weapons', 18), 'attr1': ('Pleasant', 24), 'attr2': ('Unpleasant', 24)}
weat3 {'targ1': ('MaleNames', 27), 'targ2': ('FemaleNames', 27), 'attr1': ('Pleasant', 24), 'attr2': ('Unpleasant', 24)}
weat6 {'targ1': ('MaleNames', 27), 'targ2': ('FemaleNames', 27), 'attr1': ('Career', 8), 'attr2': ('Family', 8)}
weat6b {'targ1': ('MaleTerms', 8), 'targ2': ('FemaleTerms', 8), 'attr1': ('Career', 8), 'attr2': ('Family', 8)}
weat7 {'targ1': ('Math', 17), 'targ2': ('Arts', 17), 'attr1': ('MaleTerms', 8), 'attr2': ('FemaleTerms', 8)}
weat7b {'targ1': ('Math', 17), 'targ2': ('Arts', 17), 'attr1': ('MaleNames', 27), 'attr2': ('FemaleNames', 27)}
weat8 {'targ1': ('Science', 17), 'targ2': ('Arts', 17), 'attr1': ('MaleTerms', 8), 'attr2': ('FemaleTerms', 8)}
weat8b {'targ1': ('Science', 17), 'targ2': ('Arts', 17), 'attr1': ('MaleNames', 27), 'a

In [12]:
import gc, numpy as np, torch
from transformers import AutoTokenizer, AutoModel

MODELS = {
    "banglabert"      : "csebuetnlp/banglabert",
    "banglishbert"    : "csebuetnlp/banglishbert",
    "bangla-bert-base": "sagorsarker/bangla-bert-base",
    "kowsher-bangla"  : "Kowsher/bangla-bert",
    "sahajbert"       : "neuropark/sahajBERT",
    "mbert"           : "bert-base-multilingual-cased",
    "muril"           : "google/muril-base-cased",
    "indic-bert"      : "ai4bharat/indic-bert",
    "xlm-r"           : "xlm-roberta-base",
}

# WEAT-এর gender category থেকে সব শব্দ
weat_words = set()
for c in GENDER_CATS + CONTROL_CATS:
    for s in ["targ1","targ2","attr1","attr2"]:
        weat_words.update(WEAT[c][s]["examples"])

pok_words = set()
for v in POK.values():
    pok_words.update(v)

ALL = sorted(weat_words | pok_words)
print(f"WEAT: {len(weat_words)} | Pokkhopat: {len(pok_words)} | মোট unique: {len(ALL)}")
print("multi-word:", sum(1 for w in ALL if " " in w))

device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings, frag_rows = {}, []

for name, hf in MODELS.items():
    print(f"\n{name} ...", end=" ")
    try:
        tok = AutoTokenizer.from_pretrained(hf)
        mdl = AutoModel.from_pretrained(hf).to(device).eval()
    except Exception as e:
        print("❌", str(e)[:120]); continue

    vecs, ntok = {}, {}
    for w in ALL:
        pieces = tok.tokenize(w)
        ntok[w] = len(pieces)
        inp = tok(w, return_tensors="pt").to(device)
        with torch.no_grad():
            h = mdl(**inp).last_hidden_state[0][1:-1]
        vecs[w] = h.mean(dim=0).cpu().numpy()

    embeddings[name] = vecs
    single = [w for w in ALL if " " not in w]
    multi  = [w for w in ALL if " " in w]
    frag_rows.append({
        "model": name,
        "vocab": tok.vocab_size,
        "mean_tokens_all":    round(np.mean([ntok[w] for w in ALL]), 2),
        "mean_tokens_single": round(np.mean([ntok[w] for w in single]), 2),
        "mean_tokens_multi":  round(np.mean([ntok[w] for w in multi]), 2),
        "pct_single_intact":  round(100*np.mean([ntok[w]==1 for w in single]), 1),
    })
    print("✅", frag_rows[-1]["mean_tokens_all"], "tokens/term")
    del mdl, tok; gc.collect(); torch.cuda.empty_cache()

frag_df = pd.DataFrame(frag_rows)
print("\n" + frag_df.to_string(index=False))
frag_df.to_csv("results_fragmentation.csv", index=False)

np.savez_compressed("embeddings.npz",
    **{f"{m}||{w}": v for m, d in embeddings.items() for w, v in d.items()})
print("\nsaved embeddings.npz")

WEAT: 252 | Pokkhopat: 333 | মোট unique: 556
multi-word: 87

banglabert ... 

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 3.36 tokens/term

           model  vocab  mean_tokens_all  mean_tokens_single  mean_tokens_multi  pct_single_intact
      banglabert  32000             1.69                1.46               2.92               65.9
    banglishbert  32000             1.97                1.72               3.29               51.0
bangla-bert-base 101975             2.69                2.27               4.92               35.6
  kowsher-bangla 101975             2.69                2.27               4.92               35.6
       sahajbert  32000             1.85                1.63               3.03               62.0
           mbert 119547             4.70                4.01               8.43                7.2
           muril 197258             1.98                1.77               3.10               51.2
           xlm-r 250002             3.36                2.95               5.55               15.4

saved embeddings.npz


In [13]:
frag_df.to_csv("results_fragmentation.csv", index=False)
from google.colab import files; files.download("results_fragmentation.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from transformers import AutoTokenizer
t1 = AutoTokenizer.from_pretrained("sagorsarker/bangla-bert-base")
t2 = AutoTokenizer.from_pretrained("Kowsher/bangla-bert")
v1, v2 = t1.get_vocab(), t2.get_vocab()
print("vocab একই?", v1 == v2)
print("overlap:", len(set(v1) & set(v2)), "/", len(v1))
s = "মানব সম্পদ ব্যাবস্থাপক"
print(t1.tokenize(s)); print(t2.tokenize(s))

vocab একই? True
overlap: 101975 / 101975
['মানব', 'সমপদ', 'ব', '##যা', '##বস', '##থা', '##পক']
['মানব', 'সমপদ', 'ব', '##যা', '##বস', '##থা', '##পক']


In [15]:
import numpy as np, pandas as pd
from itertools import combinations

def _cos(A,B):
    A=A/np.linalg.norm(A,axis=1,keepdims=True); B=B/np.linalg.norm(B,axis=1,keepdims=True)
    return A@B.T

def weat_d(X,Y,A,B):
    sX=_cos(X,A).mean(1)-_cos(X,B).mean(1); sY=_cos(Y,A).mean(1)-_cos(Y,B).mean(1)
    sd=np.concatenate([sX,sY]).std(ddof=1)
    return (sX.mean()-sY.mean())/sd if sd>0 else 0.0

def weat_p(X,Y,A,B,n_perm=10000,seed=0):
    rng=np.random.default_rng(seed); nX=len(X); XY=np.vstack([X,Y])
    s=_cos(XY,A).mean(1)-_cos(XY,B).mean(1); obs=s[:nX].sum()-s[nX:].sum(); n=len(s)
    if n<=20:
        combs=list(combinations(range(n),nX))
        if len(combs)<=50000:
            cnt=0
            for idx in combs:
                m=np.zeros(n,bool); m[list(idx)]=True
                if s[m].sum()-s[~m].sum()>=obs: cnt+=1
            return cnt/len(combs), "exact"
    cnt=0
    for _ in range(n_perm):
        p=rng.permutation(n)
        if s[p[:nX]].sum()-s[p[nX:]].sum()>=obs: cnt+=1
    return cnt/n_perm, "sampled"

rows=[]
for model, vecs in embeddings.items():
    for cat in CONTROL_CATS + GENDER_CATS:
        d = WEAT[cat]
        try:
            M = {s: np.array([vecs[w] for w in d[s]["examples"]])
                 for s in ["targ1","targ2","attr1","attr2"]}
        except KeyError as e:
            print(f"skip {model}/{cat}: missing {e}"); continue
        es = weat_d(M["targ1"],M["targ2"],M["attr1"],M["attr2"])
        p, mode = weat_p(M["targ1"],M["targ2"],M["attr1"],M["attr2"])
        rows.append({"model":model, "category":cat,
                     "kind":"control" if cat in CONTROL_CATS else "gender",
                     "d":round(es,3), "p":round(p,4), "mode":mode,
                     "sig":"*" if p<0.05 else ""})

weat_df = pd.DataFrame(rows)
print("=== effect size (d) ===")
print(weat_df.pivot(index="model", columns="category", values="d").to_string())
print("\n=== full ===")
print(weat_df.to_string(index=False))
weat_df.to_csv("results_weat.csv", index=False)

=== effect size (d) ===
category          weat1  weat2  weat6  weat6b  weat7  weat7b  weat8  weat8b
model                                                                      
bangla-bert-base  0.546  0.552 -0.063  -0.298 -0.324   0.122 -0.056   0.005
banglabert        0.621  0.610  0.559   0.792 -0.591  -0.780 -1.060  -1.068
banglishbert      0.760  0.797 -1.015   0.218  0.592  -0.758 -0.657  -0.754
kowsher-bangla    0.734  0.927 -0.236  -0.084  0.130  -0.988  0.369  -1.013
mbert             0.280 -0.160  0.473   0.345 -0.731   0.001 -0.619   0.037
muril             1.014  0.567 -1.155   0.515 -0.329  -0.529 -0.471  -0.807
sahajbert         0.505 -0.039  0.322   0.433 -0.241  -0.756 -0.296  -0.132
xlm-r            -0.060 -0.082  0.263   0.810  0.122  -0.102 -0.842  -0.576

=== full ===
           model category    kind      d      p    mode sig
      banglabert    weat1 control  0.621 0.0338 sampled   *
      banglabert    weat2 control  0.610 0.0335 sampled   *
      banglabert    we

In [16]:
frag_df.to_csv("results_weat.csv", index=False)
from google.colab import files; files.download("results_weat.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import unicodedata
from transformers import AutoTokenizer

tests = ["সম্পদ", "ব্যবস্থাপক", "বিদ্যুৎ", "কৃষ্ণ", "শিক্ষক", "স্বাস্থ্য"]
rows = []
for name, hf in MODELS.items():
    if name == "indic-bert": continue
    try: tok = AutoTokenizer.from_pretrained(hf)
    except Exception: continue
    lost = 0
    for w in tests:
        pieces = tok.tokenize(w)
        rebuilt = "".join(p.replace("##","").replace("▁","") for p in pieces)
        has_virama_in  = "\u09CD" in w
        has_virama_out = "\u09CD" in rebuilt
        if has_virama_in and not has_virama_out: lost += 1
        rows.append({"model":name,"word":w,"tokens":pieces,
                     "rebuilt":rebuilt,"virama_lost": has_virama_in and not has_virama_out})
    print(f"{name:18} হসন্ত হারালো {lost}/{len(tests)} শব্দে")

import pandas as pd
norm_df = pd.DataFrame(rows)
print("\n", norm_df.to_string(index=False))
norm_df.to_csv("results_normalization.csv", index=False)

banglabert         হসন্ত হারালো 0/6 শব্দে
banglishbert       হসন্ত হারালো 0/6 শব্দে
bangla-bert-base   হসন্ত হারালো 6/6 শব্দে
kowsher-bangla     হসন্ত হারালো 6/6 শব্দে
sahajbert          হসন্ত হারালো 0/6 শব্দে
mbert              হসন্ত হারালো 0/6 শব্দে
muril              হসন্ত হারালো 0/6 শব্দে
xlm-r              হসন্ত হারালো 0/6 শব্দে

            model       word                        tokens    rebuilt  virama_lost
      banglabert      সম্পদ                       [সম্পদ]      সম্পদ        False
      banglabert ব্যবস্থাপক                  [ব্যবস্থাপক] ব্যবস্থাপক        False
      banglabert    বিদ্যুৎ                     [বিদ্যুৎ]    বিদ্যুৎ        False
      banglabert      কৃষ্ণ                       [কৃষ্ণ]      কৃষ্ণ        False
      banglabert     শিক্ষক                      [শিক্ষক]     শিক্ষক        False
      banglabert  স্বাস্থ্য                   [স্বাস্থ্য]  স্বাস্থ্য        False
    banglishbert      সম্পদ                       [সম্পদ]      সম্পদ        False
    ba

In [18]:
frag_df.to_csv("results_normalization.csv", index=False)
from google.colab import files; files.download("results_normalization.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
import json, glob, os, gc, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModel

SEAT = {}
for f in sorted(glob.glob("data/BCB/seat_bn_data/data/*.jsonl")):
    SEAT[os.path.basename(f).replace(".jsonl","")] = json.load(open(f, encoding="utf-8"))
print("SEAT:", {k: {s: len(v[s]["examples"]) for s in ["targ1","targ2","attr1","attr2"]}
                for k,v in list(SEAT.items())[:2]})

device = "cuda" if torch.cuda.is_available() else "cpu"
seat_rows = []

for name, hf in MODELS.items():
    if name == "indic-bert": continue
    try:
        tok = AutoTokenizer.from_pretrained(hf)
        mdl = AutoModel.from_pretrained(hf).to(device).eval()
    except Exception as e:
        print(name, "❌", str(e)[:80]); continue

    def embed(sents):
        out = []
        for i in range(0, len(sents), 32):
            b = tok(sents[i:i+32], return_tensors="pt", padding=True, truncation=True).to(device)
            with torch.no_grad():
                h = mdl(**b).last_hidden_state
            mask = b["attention_mask"].unsqueeze(-1)
            out.append(((h*mask).sum(1)/mask.sum(1)).cpu().numpy())
        return np.vstack(out)

    for cat, d in SEAT.items():
        M = {s: embed(d[s]["examples"]) for s in ["targ1","targ2","attr1","attr2"]}
        es = weat_d(M["targ1"],M["targ2"],M["attr1"],M["attr2"])
        p, mode = weat_p(M["targ1"],M["targ2"],M["attr1"],M["attr2"], n_perm=2000)
        seat_rows.append({"model":name,"category":cat,
                          "kind":"control" if cat in ["seat1","seat2"] else "gender",
                          "d":round(es,3),"p":round(p,4),"sig":"*" if p<0.05 else ""})
    print(name, "✅")
    del mdl, tok; gc.collect(); torch.cuda.empty_cache()

seat_df = pd.DataFrame(seat_rows)
print("\n=== SEAT effect size ===")
print(seat_df.pivot(index="model", columns="category", values="d").to_string())
seat_df.to_csv("results_seat.csv", index=False)

SEAT: {'seat1': {'targ1': 119, 'targ2': 119, 'attr1': 120, 'attr2': 120}, 'seat2': {'targ1': 126, 'targ2': 126, 'attr1': 120, 'attr2': 120}}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


banglabert ✅


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


muril ✅


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


xlm-r ✅

=== SEAT effect size ===
category          seat1  seat2  seat3  seat6  seat6b  seat7  seat7b  seat8  seat8b
model                                                                             
bangla-bert-base  0.324  0.561 -0.003  0.130  -0.066  0.192   0.952  0.076   0.290
banglabert        0.298  0.283  0.291  0.119   0.065 -0.370  -0.581 -0.847  -0.746
banglishbert      0.033  0.736  0.809 -0.479   0.010  0.414  -0.602 -0.175  -0.565
kowsher-bangla    0.852  1.074 -0.402  0.768   0.103  0.404  -0.180  0.708  -0.679
mbert            -0.017 -0.136  0.171  0.443   0.248 -0.523  -0.305 -0.412   0.186
muril             1.042  1.307  0.262  0.451   0.177  0.053  -0.459  0.018   0.391
sahajbert         0.564  0.097  0.527  0.279   0.627  0.192  -0.649  0.061  -0.319
xlm-r            -0.085  0.110  0.585  0.629  -0.099  0.435  -0.638  0.485  -0.710


In [20]:
frag_df.to_csv("results_seat.csv", index=False)
from google.colab import files; files.download("results_seat.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
from google.colab import files
for f in ["results_fragmentation.csv", "results_weat.csv", "results_seat.csv"]:
    try:
        files.download(f)
    except Exception as e:
        print(f, "→", e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
import pandas as pd

# কোনটা আছে দেখে নাও
for n in ["frag_df", "weat_df", "seat_df", "norm_df"]:
    print(n, "→", globals()[n].shape if n in globals() else "নেই")

# এবার সঠিক নাম দিয়ে save
frag_df.to_csv("results_fragmentation.csv", index=False)
weat_df.to_csv("results_weat.csv", index=False)
seat_df.to_csv("results_seat.csv", index=False)
if "norm_df" in globals():
    norm_df.to_csv("results_normalization.csv", index=False)

# যাচাই — প্রতিটায় সঠিক জিনিস আছে কিনা
print("\n--- যাচাই ---")
for f, expect in [("results_fragmentation.csv", 8),
                  ("results_weat.csv", 64),
                  ("results_seat.csv", 72)]:
    d = pd.read_csv(f)
    ok = "✅" if len(d) == expect else "❌"
    print(f"{ok} {f:32} {d.shape}  cols: {list(d.columns)[:4]}")

frag_df → (8, 6)
weat_df → (64, 7)
seat_df → (72, 6)
norm_df → (48, 5)

--- যাচাই ---
✅ results_fragmentation.csv        (8, 6)  cols: ['model', 'vocab', 'mean_tokens_all', 'mean_tokens_single']
✅ results_weat.csv                 (64, 7)  cols: ['model', 'category', 'kind', 'd']
✅ results_seat.csv                 (72, 6)  cols: ['model', 'category', 'kind', 'd']


In [24]:
from google.colab import files
for f in ["results_fragmentation.csv", "results_weat.csv",
          "results_seat.csv", "results_normalization.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>